### TO DO - reunião 22/04/2026

Especificações após reunião realizada com a equipe de DIMAP-1

1. acrescentar informações dos lotes condominiais (apartamento) vinculando eles aos polígonos
2. fazer o melt dos lotes que tem mais de um polígono, transformando em multipoligono
3. criar a base indexada por Setor, Quadra, Lote, Condominio, Tipo de Lote (ou seja, pelo lote em si seja apartamento ou não) - aí os lotes de apartamento herdam o zonemento do polígono do lote do condominio
4. A base final, indexada por lote, deve ter uma coluna para cada zoneamento existente. dentro dessa coluna, um número de 1 a 100 que representa a importância daqueel zoneamento (como já calculamos) para aquele lote. A soma das importancias de todas as zonas deve ser igual a 100
5. A base final deve ser nesse formato:

| Setor | Quadra | Lote | Condomínio | Tipo de Lote | ZOE | ZC | ZEU |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 001 | 015 | 0024 | 08 | F | 20 | 30 | 50 |
| 002 | 042 | 1250 | 15 | M | 10 | 80 | 10 |
| 003 | 088 | 0500 | 22 | V | 45 | 45 | 10 |
| 004 | 102 | 9999 | 40 | F | 60 | 20 | 20 |
| 005 | 067 | 0001 | 99 | M | 33 | 33 | 34 |

6. Acrescentar na base uma coluna de observações (para falar se o lote é multipoligono, etc.) e também colocar a área total do lote e a área total intersectada pelo zoneamento

Enviar também a base indexada por id do polígono para o Bruno Carano.

In [16]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from utils.download_file_geosampa import download_file_geosampa
from utils.io.parquet import load_parquet, save_parquet
import os
from config import FILE_IPTU_GEOSAMPA, DATA_DIR, OUTPUT_DIR

Vou voltar no dataframe original (não o do dashboard) porque vamos ter que recalcular algumas coisas que é melhor fazer depois que der join com os apartamentos.

In [17]:
df = load_parquet('df_final.parquet', subfolder='microdados_lotes_com_zoneamento', gdf=False, output=True)

In [18]:
df.head()

,id_pol_lote,cd_setor_fiscal,cd_quadra_fiscal,cd_lote,cd_condominio,cd_tipo_lote,cd_zoneamento_perimetro,an_legislacao_zoneamento,area_pol_lote,area_interseccao,id_perimetro_zoneamento,percentual_interseccao,sql
0,5967194,210,008,0053,00,F,ZMa,2024,270.751684,270.751684,536791,100.0,210.008.0053-00
1,5967202,210,011,0047,00,F,ZMa,2024,176.359862,176.359862,536783,100.0,210.011.0047-00
2,5967211,210,016,0099,00,F,ZEUa,2024,150.331192,150.331192,528202,100.0,210.016.0099-00
3,5967212,210,011,0087,00,F,ZMa,2024,145.726729,145.726729,536783,100.0,210.011.0087-00
4,5967247,210,008,0110,00,F,ZMa,2024,161.244863,161.244863,536791,100.0,210.008.0110-00


In [19]:
def load_iptu()->pd.DataFrame:
    
    fname_iptu = 'iptu_2026.parquet'
    fpath_iptu = os.path.join(DATA_DIR, 'iptu', fname_iptu)
    if os.path.exists(fpath_iptu):
        print(f"Arquivo encontrado em {fpath_iptu}. Carregando...")
       
        return load_parquet(fpath_iptu, subfolder='iptu', gdf=False, output=False)
    else:
        print(f"Arquivo não encontrado em {fpath_iptu}. Baixando do GeoSampa...")
        df = download_file_geosampa(FILE_IPTU_GEOSAMPA)
        save_parquet(df, 'iptu_2026.parquet', subfolder='iptu', output=False)
        return df

In [20]:
iptu = load_iptu()

Arquivo encontrado em C:\Users\d835916\projects\zoneamento_lotes_pmsp\data\iptu\iptu_2026.parquet. Carregando...


In [21]:
iptu.head()

,NUMERO DO CONTRIBUINTE,ANO DO EXERCICIO,NUMERO DA NL,DATA DO CADASTRAMENTO,NUMERO DO CONDOMINIO,CODLOG DO IMOVEL,NOME DE LOGRADOURO DO IMOVEL,NUMERO DO IMOVEL,COMPLEMENTO DO IMOVEL,BAIRRO DO IMOVEL,...,ANO DA CONSTRUCAO CORRIGIDO,QUANTIDADE DE PAVIMENTOS,TESTADA PARA CALCULO,TIPO DE USO DO IMOVEL,TIPO DE PADRAO DA CONSTRUCAO,TIPO DE TERRENO,FATOR DE OBSOLESCENCIA,ANO DE INICIO DA VIDA DO CONTRIBUINTE,MES DE INICIO DA VIDA DO CONTRIBUINTE,FASE DO CONTRIBUINTE
0,0010030001-4,2026,1,01/01/26,00-0,03812-1,R S CAETANO,13.0,NaN,SANTA EFIGENIA,...,1924,1,13.00,Loja,Comercial horizontal - padrão B,De esquina,0.2,1963,1,0
1,0010030002-2,2026,1,01/01/26,00-0,03812-1,R S CAETANO,19.0,NaN,SANTA EFIGENIA,...,1944,1,6.00,Loja,Comercial horizontal - padrão B,Normal,0.2,1963,1,0
2,0010030003-0,2026,1,01/01/26,00-0,03812-1,R S CAETANO,27.0,NaN,SANTA EFIGENIA,...,1965,2,7.85,Loja,Comercial horizontal - padrão B,Normal,0.2,1963,1,0
3,0010030004-9,2026,1,01/01/26,00-0,03812-1,R S CAETANO,33.0,NaN,NaN,...,1944,1,6.05,Loja,Comercial horizontal - padrão B,Normal,0.2,1963,1,0
4,0010030005-7,2026,1,01/01/26,00-0,03812-1,R S CAETANO,39.0,NaN,NaN,...,2004,2,6.70,Loja,Comercial horizontal - padrão B,Normal,0.8,1963,1,0


In [22]:
iptu.shape

(3920972, 29)